In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

# Veri setini Google Drive'dan okuma
# Lütfen kopyaladığınız yolu aşağıdaki tırnak işaretleri arasına yapıştırın
df = pd.read_csv('/content/drive/MyDrive/Fraud_Project/creditcard.csv')

# Verinin ilk 5 satırını görüntüleme (Okuma işleminin başarılı olup olmadığını kontrol etmek için)
df.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [ ]:
# 1. Veri setinin toplam satır ve sütun sayısını öğrenelim
print("Veri Seti Boyutu (Satır, Sütun):", df.shape)

print("\n" + "="*45 + "\n")

# 2. Verilerde eksik (boş) değer var mı kontrol edelim
eksik_degerler = df.isnull().sum().sum()
print(f"Toplam Eksik (Boş) Değer Sayısı: {eksik_degerler}")

print("\n" + "="*45 + "\n")

# 3. Dolandırıcılık (Fraud - 1) ve Normal (0) işlemlerin sayısına bakalım
print("İşlem Sınıflarının Dağılımı (0: Normal, 1: Dolandırıcılık):")
print(df['Class'].value_counts())

Veri Seti Boyutu (Satır, Sütun): (284807, 31)


Toplam Eksik (Boş) Değer Sayısı: 0


İşlem Sınıflarının Dağılımı (0: Normal, 1: Dolandırıcılık):
Class
0    284315
1       492
Name: count, dtype: int64


In [ ]:
from sklearn.preprocessing import StandardScaler

# StandardScaler nesnesini oluşturalım (Verileri aynı ölçeğe getirmek için)
scaler = StandardScaler()

# 'Time' ve 'Amount' sütunlarını ölçeklendirelim
df['Scaled_Time'] = scaler.fit_transform(df['Time'].values.reshape(-1, 1))
df['Scaled_Amount'] = scaler.fit_transform(df['Amount'].values.reshape(-1, 1))

# Eski orijinal 'Time' ve 'Amount' sütunlarını silelim (Çünkü yenilerini oluşturduk)
df.drop(['Time', 'Amount'], axis=1, inplace=True)

# Değişiklikleri görmek için verinin ilk 5 satırını tekrar yazdıralım
df.head()

,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,...,V22,V23,V24,V25,V26,V27,V28,Class,Scaled_Time,Scaled_Amount
0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,...,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,0,-1.996583,0.244964
1,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,...,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,0,-1.996583,-0.342475
2,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,...,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,0,-1.996562,1.160686
3,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,...,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,0,-1.996562,0.140534
4,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,...,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,0,-1.996541,-0.073403


In [ ]:
from sklearn.model_selection import train_test_split

# 1. Hedef değişkeni (y) ve özellikleri (X) ayıralım
# 'Class' sütunu hedefimiz, geri kalanlar modelin öğreneceği özellikler
X = df.drop('Class', axis=1)
y = df['Class']

# 2. Veriyi Eğitim (%80) ve Test (%20) olarak bölelim
# stratify=y: Eğitim ve test setlerinde normal/dolandırıcılık oranının aynı kalmasını sağlar
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Eğitim seti boyutu (X_train):", X_train.shape)
print("Test seti boyutu (X_test):", X_test.shape)

Eğitim seti boyutu (X_train): (227845, 30)
Test seti boyutu (X_test): (56962, 30)


In [ ]:
from imblearn.over_sampling import SMOTE

# SMOTE nesnesini oluşturalım (Veri dengesizliğini çözmek için)
smote = SMOTE(random_state=42)

# Sadece Eğitim verilerine (X_train, y_train) SMOTE uygulayalım
# DİKKAT: Test verilerine (X_test) asla dokunmuyoruz! Onlar gerçek hayatı temsil ediyor.
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

# SMOTE öncesi ve sonrası durumu karşılaştıralım
print("SMOTE Öncesi Eğitim Setindeki Sınıf Dağılımı (0: Normal, 1: Dolandırıcılık):")
print(y_train.value_counts())

print("\n" + "="*45 + "\n")

print("SMOTE Sonrası Eğitim Setindeki Sınıf Dağılımı (0: Normal, 1: Dolandırıcılık):")
print(y_train_smote.value_counts())

SMOTE Öncesi Eğitim Setindeki Sınıf Dağılımı (0: Normal, 1: Dolandırıcılık):
Class
0    227451
1       394
Name: count, dtype: int64


SMOTE Sonrası Eğitim Setindeki Sınıf Dağılımı (0: Normal, 1: Dolandırıcılık):
Class
0    227451
1    227451
Name: count, dtype: int64


In [ ]:
from sklearn.ensemble import RandomForestClassifier

# 1. Random Forest modelini oluşturalım
# n_estimators=100: 100 farklı karar ağacı kullanacağız
# n_jobs=-1: Bilgisayarın tüm işlemci gücünü kullanarak işlemi hızlandırır
model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# 2. Modeli, SMOTE ile dengelediğimiz verilerle eğitelim
# Bu işlem veri boyutu büyük olduğu için 1-2 dakika sürebilir
print("Model eğitiliyor, lütfen bekleyin...")
model.fit(X_train_smote, y_train_smote)

print("Model başarıyla eğitildi!")

Model eğitiliyor, lütfen bekleyin...
Model başarıyla eğitildi!


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Test verileri üzerinde tahmin yapalım
y_pred = model.predict(X_test)

# 2. Sonuçları raporlayalım
print("--- Model Performans Raporu ---")
print(classification_report(y_test, y_pred))

print("\n--- Karışıklık Matrisi (Confusion Matrix) ---")
print(confusion_matrix(y_test, y_pred))

--- Model Performans Raporu ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.81      0.82        98

    accuracy                           1.00     56962
   macro avg       0.92      0.90      0.91     56962
weighted avg       1.00      1.00      1.00     56962


--- Karışıklık Matrisi (Confusion Matrix) ---
[[56849    15]
 [   19    79]]
